# 03 Matching and Prediction

This notebook uses real held-out trajectories to inspect neighbor retrieval, history reranking, one-step forecasts, and short rollouts. The point is not only to show the API, but to reveal where the real data violate modeling assumptions.


## Goals

1. Build a real-data reference bank and hold out at least one real embryo for queries.
2. Compare position-only and history-aware matching on the held-out embryo.
3. Inspect the local increment cloud selected by the model.
4. Run a short rollout and compare it against the observed future on the real trajectory.


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "dev" / "particle_prediction" / "data" / "loading.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dev.particle_prediction.data.loading import DEFAULT_BUILD_DIR, FILE_PREFIX, load_trajectories
from dev.particle_prediction.data.smoothing import smooth_trajectory, smooth_trajectories
from dev.particle_prediction.data.resampling import compute_cumulative_arc_length, resample_smoothed_trajectory, resample_smoothed_trajectories
from dev.particle_prediction.data.transition_windows import build_transition_windows
from dev.particle_prediction.data.transition_bank import build_transition_bank
from dev.particle_prediction.data.dataset import build_prediction_tasks, build_query_from_resampled_trajectory
from dev.particle_prediction.eval.evaluate import comparison_table, run_evaluation_suite
from dev.particle_prediction.models.local_transition_pf import LocalTransitionPredictor
from dev.particle_prediction.models.matching import MatchingConfig, compare_matching_modes
from dev.particle_prediction.viz.smoothing import (
    plot_latent_trajectory_before_after_smoothing,
    plot_raw_vs_smoothed_timeseries,
    plot_sg_parameter_sweep,
)
from dev.particle_prediction.viz.transition_bank import (
    plot_arc_length_vs_time,
    plot_bank_state_density,
    plot_history_segments_example,
    plot_increment_norm_distribution,
    plot_resampled_points_on_trajectory,
    plot_transition_windows_for_embryo,
)
from dev.particle_prediction.viz.matching import (
    compare_default_vs_fast_matching,
    plot_history_offset_heatmap,
    plot_history_reranking,
    plot_query_and_candidate_neighbors,
)
from dev.particle_prediction.viz.prediction import (
    plot_local_increment_cloud,
    plot_prediction_fan,
    plot_rollout_against_truth,
    plot_sampled_next_steps,
    plot_support_diagnostics_along_rollout,
)
from dev.particle_prediction.viz.evaluation import (
    plot_error_vs_horizon,
    plot_error_vs_support,
    plot_failure_gallery,
    plot_model_comparison_table,
)

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)

def resolve_real_data_context(max_default_trajectories=8):
    build_dir = Path(os.environ.get("MORPHSEQ_PARTICLE_BUILD_DIR", project_root / DEFAULT_BUILD_DIR))
    if not build_dir.exists():
        raise FileNotFoundError(
            f"Expected real data under {build_dir}. Set MORPHSEQ_PARTICLE_BUILD_DIR to override."
        )

    available_experiments = sorted(
        path.stem[len(FILE_PREFIX):]
        for path in build_dir.glob(f"{FILE_PREFIX}*.csv")
    )
    if not available_experiments:
        raise FileNotFoundError(f"No {FILE_PREFIX}*.csv files found in {build_dir}")

    env_experiments = os.environ.get("MORPHSEQ_PARTICLE_EXPERIMENTS")
    experiment_ids = [part.strip() for part in env_experiments.split(",") if part.strip()] if env_experiments else [available_experiments[0]]
    env_limit = os.environ.get("MORPHSEQ_PARTICLE_MAX_TRAJECTORIES")
    max_trajectories = int(env_limit) if env_limit else max_default_trajectories
    return build_dir, available_experiments, experiment_ids, max_trajectories

build_dir, available_experiments, experiment_ids, max_trajectories = resolve_real_data_context()
print(f"Project root: {project_root}")
print(f"Build dir: {build_dir}")
print(f"Available experiments: {available_experiments[:5]}{' ...' if len(available_experiments) > 5 else ''}")
print(f"Notebook experiments: {experiment_ids}")
print(f"Trajectory cap: {max_trajectories}")


In [ ]:
dataset = load_trajectories(
    build_dir=build_dir,
    experiment_ids=experiment_ids,
    n_components=10,
    scale=True,
    min_trajectory_length=5,
    verbose=False,
)
selected_trajectories = dataset.trajectories[:max_trajectories]
summary_df = pd.DataFrame(
    {
        "embryo_id": [traj.embryo_id for traj in selected_trajectories],
        "experiment_id": [traj.experiment_id for traj in selected_trajectories],
        "perturbation_class": [traj.perturbation_class for traj in selected_trajectories],
        "n_frames": [len(traj.time_seconds) for traj in selected_trajectories],
        "delta_t": [traj.delta_t for traj in selected_trajectories],
        "n_hard_gaps": [int(np.sum(traj.hard_gap_mask)) if traj.hard_gap_mask is not None else 0 for traj in selected_trajectories],
        "n_interpolatable_gaps": [int(np.sum(traj.interpolatable_gap_mask)) if traj.interpolatable_gap_mask is not None else 0 for traj in selected_trajectories],
    }
)
print(f"Loaded {len(dataset.trajectories)} trajectories; using first {len(selected_trajectories)} for the notebook walkthrough.")
summary_df.head(10)


In [ ]:
window_seconds = 5.0 * 3600.0
poly_order = 2
delta_s = 0.25
history_length = 5
smoothed_subset = smooth_trajectories(selected_trajectories, window_seconds=window_seconds, poly_order=poly_order)
resampled_subset = resample_smoothed_trajectories(smoothed_subset, delta_s=delta_s)
if len(resampled_subset) < 3:
    raise ValueError("Need at least three trajectories for a simple holdout walkthrough.")
reference_trajectories = resampled_subset[:-1]
heldout_trajectory = resampled_subset[-1]
bank = build_transition_bank(reference_trajectories, history_length=history_length + 2, use_state_index=True)
print(f"Reference trajectories: {len(reference_trajectories)} | Held out: {heldout_trajectory.source.embryo_id} | Bank windows: {len(bank)}")


## Build Real Queries from the Held-Out Embryo


In [ ]:
state_index = min(max(history_length + 2, 8), len(heldout_trajectory.resampled) - 4)
snapshot_query = build_query_from_resampled_trajectory(heldout_trajectory, state_index=state_index, history_length=None)
history_query = build_query_from_resampled_trajectory(heldout_trajectory, state_index=state_index, history_length=history_length)
true_future = heldout_trajectory.resampled[state_index + 1 : state_index + 4]
context_points = heldout_trajectory.resampled[state_index - history_length : state_index + 1]
print(snapshot_query.metadata)
print(history_query.metadata)


## Matching Diagnostics on Real Data


In [ ]:
matches = compare_matching_modes(bank, history_query.current_state, history_query.history_segments)
plot_query_and_candidate_neighbors(history_query.current_state, matches["default"])


In [ ]:
plot_history_reranking(matches["default"], matches["fast_summary"])


In [ ]:
plot_history_offset_heatmap(history_query.history_segments, bank.windows[matches["default"].candidate_indices[0]], offset_radius=1)


In [ ]:
compare_default_vs_fast_matching(matches["default"], matches["fast_summary"])


## One-Step and Rollout Prediction


In [ ]:
predictor = LocalTransitionPredictor(
    bank=bank,
    matching_config=MatchingConfig(k_state=32, offset_radius=1, retrieval_method="nn"),
    sigma_parallel=0.05,
    sigma_perp=0.1,
    jitter_mode="tangent",
)
one_step = predictor.predict_query(history_query, n_samples=64, rng=np.random.default_rng(7))
rollout = predictor.rollout_query(history_query, n_steps=3, n_particles=64, rng=np.random.default_rng(11))
one_step.diagnostics


In [ ]:
plot_local_increment_cloud(history_query.current_state, one_step)


In [ ]:
plot_sampled_next_steps(history_query.current_state, one_step, true_next_state=true_future[0])


In [ ]:
plot_prediction_fan(history_query.current_state, rollout, true_future=true_future)


In [ ]:
plot_rollout_against_truth(history_query.current_state, rollout, true_future=true_future, context_points=context_points)


In [ ]:
plot_support_diagnostics_along_rollout(rollout)


## Real-Data QC Questions

- Does the held-out embryo live in a sparse part of latent space where the candidate pool becomes weak?
- Are the selected neighbors biologically plausible, or are they matching to obvious artifacts?
- Does the rollout drift off-manifold or simply follow the wrong branch?
